# Introduction to Modular, Configuration-Driven PyTorch Model Training with ***torchdistill***

## 0. Setup

### Install torchdistill

In [ ]:
!pip install torchdistill

### Download example configs

In [ ]:
!git clone https://gist.github.com/yoshitomo-matsubara/24feff8581886225b81191b9e60799c5.git examples

## 1. Play with declarative PyYAML configurations

**Key ideas**
- Design a training pipeline with abstract modules/components
- Define and instantiate the modules/components by a configuration file

→ Generalize the training pipeline and design an experiment by **your configuration file**  
→ Your configuration file explains ***almost everything*** about the experiment

### Example 1: Define CIFAR-10 dataset

See [torchvision.datasets.CIFAR10](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR10.html) documentation.

**Hardcoded ver.**

In [ ]:
import torchvision
hardcoded_config = {
    'dataset': torchvision.datasets.CIFAR10(
        root='./cifar10',
        train=True,
        download=True
    )
}
print(hardcoded_config)

**torchdistill ver.**  
You should see the same output as above

In [ ]:
from torchdistill.common import yaml_util
config = yaml_util.load_yaml_file('./examples/cifar10_v1.yaml')
print(config)

Check `./examples/cifar10_v1.yaml` by `!cat ./examples/cifar10_v1.yaml`.  
It should look as shown below.

```yaml
dataset: !import_call
  key: torchvision.datasets.CIFAR10
  init:
    kwargs:
      root: ./cifar10
      train: True
      download: True
```


### Example 2: Define CIFAR-10 dataset with transform
Define `transform` for preprocessing input images

**Hardcoded ver.**

In [ ]:
import torchvision
hardcoded_config = {
    'dataset': torchvision.datasets.CIFAR10(
        root='./cifar10',
        train=True,
        download=True,
        transform=torchvision.transforms.Compose(
            [
                torchvision.transforms.RandomCrop(
                    size=32,
                    padding=4
                ),
                torchvision.transforms.RandomHorizontalFlip(
                    p=0.5
                ),
                torchvision.transforms.ToTensor(),
                torchvision.transforms.Normalize(
                    mean=[0.49139968, 0.48215841, 0.44653091],
                    std=[0.24703223, 0.24348513, 0.26158784]
                )
            ]
        )
    )
}
print(hardcoded_config)

**torchdistill ver.**  
You should see the same output as above

In [ ]:
from torchdistill.common import yaml_util
config = yaml_util.load_yaml_file('./examples/cifar10_v2.yaml')
print(config)

Check `./examples/cifar10_v2.yaml` by `!cat ./examples/cifar10_v2.yaml`.  
It should look as shown below.

```yaml
dataset: !import_call
  key: torchvision.datasets.CIFAR10
  init:
    kwargs:
      root: ./cifar10
      train: True
      download: True
      transform: !import_call
        key: torchvision.transforms.Compose
        init:
          kwargs:
            transforms:
              - !import_call
                key: torchvision.transforms.RandomCrop
                init:
                  kwargs:
                    size: 32
                    padding: 4
              - !import_call
                key: torchvision.transforms.RandomHorizontalFlip
                init:
                  kwargs:
                    p: 0.5
              - !import_call
                key: torchvision.transforms.ToTensor
                init:
              - !import_call
                key: torchvision.transforms.Normalize
                init:
                  kwargs:
                    mean: [0.49139968, 0.48215841, 0.44653091]
                    std: [0.24703223, 0.24348513, 0.26158784]
```


### Example 3: Define CIFAR-10 training and test datasets

Reuse some of the parameters used in the training dataset for the test dataset

**Hardcoded ver.**

In [ ]:
import torchvision
hardcoded_config = {
    'datasets': {
        'train': torchvision.datasets.CIFAR10(
            root='./cifar10',
            train=True,
            download=True,
            transform=torchvision.transforms.Compose(
                [
                    torchvision.transforms.RandomCrop(
                        size=32,
                        padding=4
                    ),
                    torchvision.transforms.RandomHorizontalFlip(
                        p=0.5
                    ),
                    torchvision.transforms.ToTensor(),
                    torchvision.transforms.Normalize(
                        mean=[0.49139968, 0.48215841, 0.44653091],
                        std=[0.24703223, 0.24348513, 0.26158784]
                    )
                ]
            )
        ),
        'test': torchvision.datasets.CIFAR10(
            root='./cifar10',
            train=False,
            download=True,
            transform=torchvision.transforms.Compose(
                [
                    torchvision.transforms.ToTensor(),
                    torchvision.transforms.Normalize(
                        mean=[0.49139968, 0.48215841, 0.44653091],
                        std=[0.24703223, 0.24348513, 0.26158784]
                    )
                ]
            )
        )
    }
}
print(hardcoded_config)

**torchdistill ver.**  
You should see the same output as above

In [ ]:
from torchdistill.common import yaml_util
config = yaml_util.load_yaml_file('./examples/cifar10_v3.yaml')
print(config)

Check `./examples/cifar10_v3.yaml` by `!cat ./examples/cifar10_v3.yaml`.  
It should look as shown below, leveraging YAML anchor (`&`) and reference (`*`) to reuse previously defined values

```yaml
datasets:
  train: !import_call
    key: &dataset_key torchvision.datasets.CIFAR10
    init:
      kwargs:
        root: &root_dir ./cifar10
        train: True
        download: True
        transform: !import_call
          key: torchvision.transforms.Compose
          init:
            kwargs:
              transforms:
                - !import_call
                  key: torchvision.transforms.RandomCrop
                  init:
                    kwargs:
                      size: 32
                      padding: 4
                - !import_call
                  key: torchvision.transforms.RandomHorizontalFlip
                  init:
                    kwargs:
                      p: 0.5
                - !import_call
                  key: torchvision.transforms.ToTensor
                  init:
                - !import_call
                  key: torchvision.transforms.Normalize
                  init:
                    kwargs: &normalize_kwargs
                      mean: [0.49139968, 0.48215841, 0.44653091]
                      std: [0.24703223, 0.24348513, 0.26158784]
  test: !import_call
    key: *dataset_key
    init:
      kwargs:
        root: *root_dir
        train: False
        download: True
        transform: !import_call
          key: torchvision.transforms.Compose
          init:
            kwargs:
              transforms:
                - !import_call
                  key: torchvision.transforms.ToTensor
                  init:
                - !import_call
                  key: torchvision.transforms.Normalize
                  init:
                    kwargs: *normalize_kwargs
```

1. `key: &dataset_key torchvision.datasets.CIFAR10` records a string `'torchvision.datasets.CIFAR10'` as `dataset_key` anchor and as a value of `key` field
2. `root: &root_dir ./cifar10` records a string `'./cifar10'` as `root_dir` anchor and as a value of `root` field
3. `kwargs: &normalize_kwargs ...` records a dict `{'mean': ..., 'std': ...}` as `normalize_kwargs` anchor and as a value of `kwargs` field
4. `key: *dataset_key` inserts the `dataset_key` anchor, which is `'torchvision.datasets.CIFAR10'`
5. `root: *root_dir` inserts the `root_dir` anchor, which is `'./cifar10'`
6. `kwargs: *normalize_kwargs` inserts the `normalize_kwargs` anchor, which is `{'mean': ..., 'std': ...}`

### Example 4: Define more modules to start training a model

Define the following modules
- training data loader
- evaluation data loader
- model to train
- criterion (loss module)
- number of epochs
- optimizer
- lr scheduler

**Hardcoded ver.**

In [ ]:
import torch
import torchvision
import torchdistill.models
hardcoded_config = {
    'data_loaders': {
        'train': torch.utils.data.DataLoader(
            dataset=torchvision.datasets.CIFAR10(
                root='./cifar10',
                train=True,
                download=True,
                transform=torchvision.transforms.Compose(
                    [
                        torchvision.transforms.RandomCrop(
                            size=32,
                            padding=4
                        ),
                        torchvision.transforms.RandomHorizontalFlip(
                            p=0.5
                        ),
                        torchvision.transforms.ToTensor(),
                        torchvision.transforms.Normalize(
                            mean=[0.49139968, 0.48215841, 0.44653091],
                            std=[0.24703223, 0.24348513, 0.26158784]
                        )
                    ]
                )
            ),
            batch_size=128,
            shuffle=True,
            pin_memory=True
        ),
        'eval': torch.utils.data.DataLoader(
            dataset=torchvision.datasets.CIFAR10(
                root='./cifar10',
                train=False,
                download=True,
                transform=torchvision.transforms.Compose(
                    [
                        torchvision.transforms.ToTensor(),
                        torchvision.transforms.Normalize(
                            mean=[0.49139968, 0.48215841, 0.44653091],
                            std=[0.24703223, 0.24348513, 0.26158784]
                        )
                    ]
                )
            ),
            batch_size=256,
            shuffle=False,
            pin_memory=True
        )
    },
    'model': torchdistill.models.classification.resnet.resnet20(
        num_classes=10,
        pretrained=False
    ),
    'criterion': torch.nn.CrossEntropyLoss(
        reduction='mean'
    ),
    'num_epochs': 10,
    'optimizer': {
        'class_or_func': torch.optim.SGD,
        'kwargs': {
            'lr': 0.1
        }
    },
    'lr_scheduler': {
        'class_or_func': torch.optim.lr_scheduler.StepLR,
        'kwargs': {
            'step_size': 4,
            'gamma': 0.1
        }
    }
}
print(hardcoded_config)

**torchdistill ver.**  
You should see the same output as above

In [ ]:
from torchdistill.common import yaml_util
config = yaml_util.load_yaml_file('./examples/cifar10_v4.yaml')
print(config)

Check `./examples/cifar10_v4.yaml` by `!cat ./examples/cifar10_v4.yaml`.  
It should look as shown below

```yaml
data_loaders:
  train: !import_call
    key: torch.utils.data.DataLoader
    init:
      kwargs:
        dataset: !import_call
          key: &dataset_key torchvision.datasets.CIFAR10
          init:
            kwargs:
              root: &root_dir ./cifar10
              train: True
              download: True
              transform: !import_call
                key: torchvision.transforms.Compose
                init:
                  kwargs:
                    transforms:
                      - !import_call
                        key: torchvision.transforms.RandomCrop
                        init:
                          kwargs:
                            size: 32
                            padding: 4
                      - !import_call
                        key: torchvision.transforms.RandomHorizontalFlip
                        init:
                          kwargs:
                            p: 0.5
                      - !import_call
                        key: torchvision.transforms.ToTensor
                        init:
                      - !import_call
                        key: torchvision.transforms.Normalize
                        init:
                          kwargs: &normalize_kwargs
                            mean: [0.49139968, 0.48215841, 0.44653091]
                            std: [0.24703223, 0.24348513, 0.26158784]
        batch_size: 128
        shuffle: True
        pin_memory: True
  eval: !import_call
    key: torch.utils.data.DataLoader
    init:
      kwargs:
        dataset: !import_call
          key: *dataset_key
          init:
            kwargs:
              root: *root_dir
              train: False
              download: True
              transform: !import_call
                key: torchvision.transforms.Compose
                init:
                  kwargs:
                    transforms:
                      - !import_call
                        key: torchvision.transforms.ToTensor
                        init:
                      - !import_call
                        key: torchvision.transforms.Normalize
                        init:
                          kwargs: *normalize_kwargs
        batch_size: 256
        shuffle: False
        pin_memory: True

model: !import_call
  key: torchdistill.models.classification.resnet.resnet20
  init:
    kwargs:
      num_classes: 10
      pretrained: False

criterion: !import_call
  key: torch.nn.CrossEntropyLoss
  init:
    kwargs:
      reduction: mean

num_epochs: 10

optimizer:
  class_or_func: !import_get
    key: torch.optim.SGD
  kwargs:
    lr: 0.1

lr_scheduler:
  class_or_func: !import_get
    key: torch.optim.lr_scheduler.StepLR
  kwargs:
    step_size: 4
    gamma: 0.1
```

## 2. Train an image classification model using a declarative PyYAML configuration

### Make sure you have access to GPU
Google Colab: *Runtime* -> *Change runtime type* -> *Hardware accelarator*: "GPU"

In [ ]:
!nvidia-smi

In [ ]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm


def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_count = 0
    total_count = 0

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total_count += targets.size(0)
            correct_count += predicted.eq(targets).sum().item()

    accuracy = 100. * correct_count / total_count
    return running_loss / total_count, accuracy


def train_model(
    model, train_loader, eval_loader, criterion, optimizer, lr_scheduler, num_epochs
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        # Use tqdm for a progress bar during training
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for inputs, targets in pbar:
            inputs, targets = inputs.to(device), targets.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward + Backward + Optimize
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            pbar.set_postfix({'loss': running_loss / (pbar.n + 1)})

        # Step the learning rate scheduler if provided
        if lr_scheduler:
            lr_scheduler.step()

        # Optional validation
        if eval_loader:
            eval_loss, eval_acc = evaluate(model, eval_loader, criterion, device)
            print(f'Evaluation -> Loss: {eval_loss:.4f} | Acc: {eval_acc:.2f}%')

    print('Training Complete.')
    return model

In [ ]:
from torchdistill.common import yaml_util

config = yaml_util.load_yaml_file('./examples/cifar10_v4.yaml')
# Extract modules from config
model = config['model']

data_loaders = config['data_loaders']
train_loader = data_loaders['train']
eval_loader = data_loaders['eval']

criterion = config['criterion']

# either optimizer or lr_scheduler is not instantiated in config
optimizer_config = config['optimizer']
optimizer = optimizer_config['class_or_func'](model.parameters(), **optimizer_config['kwargs'])
lr_scheduler_config = config['lr_scheduler']
lr_scheduler = lr_scheduler_config['class_or_func'](optimizer, **lr_scheduler_config['kwargs'])

num_epochs = config['num_epochs']
model = train_model(
    model, train_loader, eval_loader, criterion, optimizer, lr_scheduler, num_epochs
)

## 3. More sample configurations, models, datasets...
For CIFAR-10/100 datasets, you can find more [sample configurations](https://github.com/yoshitomo-matsubara/torchdistill/tree/master/configs/legacy/sample/) and [models](https://github.com/yoshitomo-matsubara/torchdistill/tree/master/torchdistill/models/classification) in the [***torchdistill***](https://github.com/yoshitomo-matsubara/torchdistill) repository.
If you would like to use larger datasets e.g., **ImageNet** and **COCO** datasets and models in `torchvision` (or your own modules), refer to the [official configurations](https://github.com/yoshitomo-matsubara/torchdistill/tree/master/configs/legacy/official) used in some published papers.
Experiments with such large datasets and models will require you to use your own machine due to limited disk space and session time (12 hours for free version and 24 hours for Colab Pro) on Google Colab.
